In [0]:
# import libraries

from pyspark.sql import SparkSession, functions as F

from pyspark.sql.functions import (
    explode, desc,  row_number, col, try_divide, year, try_to_date, count, 
    countDistinct
)

from pyspark.sql.window import Window

# import pandas as pd


In [0]:
# curl https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json
# record a file in local wget -O steam_game_output.json https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json

In [0]:
# from pyspark.sql import SparkSession

filepath = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

df = spark.read.format('json').load(filepath)

In [0]:
# Number of elements in dataframe
print(f"Entries number : {df.count()}")


Entries number : 55691


The dataset is to big for json_normalize method.

In [0]:
type(df)

pyspark.sql.connect.dataframe.DataFrame

In [0]:
df.take(1)

[Row(data=Row(appid=10, categories=['Multi-player', 'Valve Anti-Cheat enabled', 'Online PvP', 'Shared/Split Screen PvP', 'PvP'], ccu=13990, developer='Valve', discount='0', genre='Action', header_image='https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513', initialprice='999', languages='English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean', name='Counter-Strike', negative=5199, owners='10,000,000 .. 20,000,000', platforms=Row(linux=True, mac=True, windows=True), positive=201215, price='999', publisher='Valve', release_date='2000/11/1', required_age='0', short_description="Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.", tags=Row(1980s=266, 1990's=1191, 2.5

In [0]:
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

We observe 23 different values : data is a only a node, categories, platforms and tags contain nested informations. Let's flatened the dataframe to range datas at same level. 

In [0]:
from pyspark.sql.functions import concat_ws

flat_df = df.select("id","data.*", concat_ws(", ", "data.categories").alias("category"))
flat_df.limit(1).display()

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website,category
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,"Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP"


In [0]:
# from pyspark.sql import functions as F

flat_df = flat_df.withColumn(
    "platform",
    F.concat_ws(', ', *[F.when(F.col(f"platforms.{f}"), F.lit(f)) for f in ['linux', 'mac', 'windows']])
)
flat_df.limit(1).display()

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website,category,platform
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,"Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP","linux, mac, windows"


In [0]:
flat_df = flat_df.withColumn("tag", F.to_json(F.col("tags")))
flat_df.limit(1).display()

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website,category,platform,tag
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,"Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP","linux, mac, windows","{""1980s"":266,""1990's"":1191,""Action"":5426,""Assassin"":227,""Classic"":2784,""Competitive"":1607,""FPS"":4831,""First-Person"":1707,""Military"":632,""Multiplayer"":3392,""Nostalgia"":131,""Old School"":769,""PvP"":881,""Score Attack"":289,""Shooter"":3353,""Strategy"":614,""Survival"":304,""Tactical""

In [0]:
flat_df = flat_df.drop( "categories", "platforms", "tags")

In [0]:
flat_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- appid: long (nullable = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- type: string (nullable = true)
 |-- website: string (nullable = true)
 |-- category: string (nullable = false)
 |-- platform: string (nullable = false)
 |-- tag: string (nullable = true)



In [0]:
# flat_df.write.csv("df.csv", header=True)

Export a csv with the .write method requires missing write right on working directory in databricks free edition. So, let's use pandas method .tocsv()

In [0]:
# import pandas as pd

# flat_df.toPandas().to_csv("df.csv", index=False)

# Explorary dataset analysis

## 1. Explore dataset

In [0]:
len(flat_df.columns)

23

In [0]:
flat_df.count()

55691

The dataframe has 23 columns and 55691 rows.

Now, let's query on the dataframe. Spark lets us run classic SQL queries on your tables, however, using classic SQL in Spark requires you to load the data in memory before running any query. We will use the .createOrReplaceTempView Spark DataFrame method in order to load the data in memory under a certain table name, we will then be able to run SQL queries on it.

In [0]:
flat_df.createOrReplaceTempView('temp_table') # Creates a temporary view table in memory called temp_table
result = spark.sql("select * from temp_table limit 1")
result.show()

+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| id|appid|  ccu|developer|discount| genre|        header_image|initialprice|           languages|          name|negative|              owners|positive|price|publisher|release_date|required_age|   short_description|type|website|            category|           platform|                 tag|
+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| 10|   10|13990|    Valve|       0|Action|https://cdn.akama...|         999|English, French, ...|Counter-Strike|    5199|10,00

The .sql method lets you write queries in SQL while benefiting from the distributed computing advantages of Spark.

In [0]:
result = spark.sql("SELECT * FROM temp_table LIMIT 1") # filters elements from temp_table 
# show everything
result.show()

+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| id|appid|  ccu|developer|discount| genre|        header_image|initialprice|           languages|          name|negative|              owners|positive|price|publisher|release_date|required_age|   short_description|type|website|            category|           platform|                 tag|
+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| 10|   10|13990|    Valve|       0|Action|https://cdn.akama...|         999|English, French, ...|Counter-Strike|    5199|10,00

We observe 
- ccu nombre de joueurs qui jouaient simulatanément au moment où le dataset a été construit
- discount en pourcentage
- prix stocké en centimes de dollars
- owners indique un intervalle pour évaluer le nombre d'acheteurs
- categories la liste varie selon le jeu. Elles permettent aux acheteurs de classer les jeux et fonctionnent comme les étagères d'une bibliothèque.
- duplicated rows ?

some problems :
- unused spaces : sort columns to verify wether it's a display problem or requests problem 
- different spelling : lowercase string columns
- special alphabetic characters on string columns developers '---', "flyingcubicle, - ", "((no-end-parens studio", "revday studio", "+7 software" , "+mpact games, llc."  
- empty entries : '' on website and platform -> verify other columns
- unused columns : id and appid are identifyers, we use name and can delete them, website, header_image 
- des colonnes numériques au format string -> convertir price, initialprice, discount, required_age au format long
- release_date string for datetime

We must verify with steam team before treat these data.
- special alphabetic characters : "," is a separator that we can use to count entries on developers, genre, languages. 
Are " ", None, ., CD PROJEKT RED, [2.21] real publishers and '---', "flyingcubicle, - ", "((no-end-parens studio", "revday studio", "+7 software" , "+mpact games, llc." real developers ?
Are "(none)", " ", "-" real publishers ?
- asiatic characters. We need information even if the langage changes.

Let's sample and observe values on selected columns.

In [0]:
flat_df.select('developer', 'publisher', 'owners', 'ccu', 'website', 'header_image').sample(fraction=0.0001).distinct().show(truncate=False)

+------------------+------------------+------------------+---+---------------------------------------------+-----------------------------------------------------------------------------+
|developer         |publisher         |owners            |ccu|website                                      |header_image                                                                 |
+------------------+------------------+------------------+---+---------------------------------------------+-----------------------------------------------------------------------------+
|Firaxis Games     |2K                |100,000 .. 200,000|34 |https://www.2k.com/games/sid-meiers-starships|https://cdn.akamai.steamstatic.com/steam/apps/282210/header.jpg?t=1568757082 |
|Lappi Soft        |Lappi Soft        |100,000 .. 200,000|0  |https://www.lappisoft.com/                   |https://cdn.akamai.steamstatic.com/steam/apps/1379270/header.jpg?t=1664398397|
|Mikage Productions|Mikage Productions|0 .. 20,000       |0  |htt

In [0]:
spark.sql("select * from temp_table limit 5").show()

+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+-------------------------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|  ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|                    short_description|type|             website|            category|           platform|                 tag|
+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------

In [0]:
result = spark.sql("select distinct type from temp_table")
result.show()

+--------+
|    type|
+--------+
|hardware|
|    game|
+--------+



In [0]:
result = spark.sql("select * from temp_table where type = 'hardware'")
result.show(truncate=False)

+------+------+---+---------+--------+-----+----------------------------------------------------------------------------+------------+---------+----------+--------+--------------------+--------+-----+-----------+------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+-----------------------------------------+---------------------------------------------+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id    |appid |ccu|developer|discount|genre|header_image                                                                |initialprice|languages|name      

Steam link est un outil édité par Anima Locus qui permet d'accéder à un jeu par lien sur le téléphone, une tablette, un autre PC et même d'accéder à un jeu hébergé sur le pc d'un ami. Ce n'est pas un jeu en soi. Il convient de supprimer la colonne type.

In [0]:
result = spark.sql("select distinct name from temp_table where name like'Counter-Strike%'")
result.show(truncate=False)

+--------------------------------+
|name                            |
+--------------------------------+
|Counter-Strike Nexon: Studio    |
|Counter-Strike: Global Offensive|
|Counter-Strike: Source          |
|Counter-Strike: Condition Zero  |
|Counter-Strike                  |
+--------------------------------+



In [0]:
from pyspark.sql.functions import col

string_cols = [field.name for field in flat_df.schema.fields if field.dataType.simpleString() == 'string']

for c in string_cols:
    count = flat_df.filter(col(c) == '').count()
    if count > 0:
        print(f"{c}: {count}")

developer: 127
genre: 161
languages: 11
publisher: 132
release_date: 99
short_description: 37
website: 25217
category: 970


In [0]:
result = spark.sql("select * from temp_table where release_date = ''")
result.show()

+-------+-------+---+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|   short_description|type|             website|            category|           platform|                 tag|
+-------+-------+---+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+-------------

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

all_cols = flat_df.columns
string_cols = [f.name for f in flat_df.schema.fields if f.dataType.simpleString() == 'string']

flat_df.select([spark_sum((col(c).isNull() | ((col(c) == '') if c in string_cols else False)).cast('int')).alias(c) for c in all_cols]).show()

+---+-----+---+---------+--------+-----+------------+------------+---------+----+--------+------+--------+-----+---------+------------+------------+-----------------+----+-------+--------+--------+---+
| id|appid|ccu|developer|discount|genre|header_image|initialprice|languages|name|negative|owners|positive|price|publisher|release_date|required_age|short_description|type|website|category|platform|tag|
+---+-----+---+---------+--------+-----+------------+------------+---------+----+--------+------+--------+-----+---------+------------+------------+-----------------+----+-------+--------+--------+---+
|  0|    0|  0|      127|       0|  161|           0|           0|       11|   0|       0|     0|       0|    0|      132|          99|           0|               37|   0|  25217|     970|       0|  0|
+---+-----+---+---------+--------+-----+------------+------------+---------+----+--------+------+--------+-----+---------+------------+------------+-----------------+----+-------+--------+----

There are 55691 rows. Website misses on one half rows. Other missing values should be converted.


In [0]:
flat_df.count() == flat_df.dropDuplicates().count()

True

In [0]:
(flat_df.count() - flat_df.dropDuplicates().count())

0

There is no duplicated value.

## 2. Clean data

Let's record clean_data under a new data_frame and create a new temp view.

In [0]:
clean_df = flat_df
clean_df.createOrReplaceTempView('table')

In [0]:
result = clean_df.select('required_age').distinct().orderBy('required_age')
result.display()

required_age
0
10
12
13
14
15
16
17
18
180


Plusieurs valeurs aberrantes : 180 et 35. Dans le monde, les jeux sont autorisés à partir de 18 ans https://fr.wikipedia.org/wiki/Syst%C3%A8me_d%27%C3%A9valuation_des_jeux_vid%C3%A9o sauf pour l'arabie saoudite et la corée à 21 ans. MA15+ est une notation australienne qui signifie autorisé à partir de 15 ans. 
Arbitrages : 
- 180 et 35 ramenés à 18 ans
- Nous allons standardiser tous les numéros suivis de + dans la mesure où les jeux sont autorisés à partir de l'âge indiqué. 

In [0]:
rules = [
    (r'\+|MA |180|35')
    ]
rule = "|".join(rules)
result = clean_df \
    .filter(col("required_age").cast('string').rlike(rule)) \
    .select('required_age') \
    .distinct() \
    .orderBy('required_age')

display(result)

required_age
180
21+
35
7+
MA 15+


In [0]:
from pyspark.sql.functions import regexp_replace, col

rules = [
    (r'\+|MA ', ''),
    (r'35|180', '18')
]

for search, replace in rules:
    clean_df = clean_df.withColumn("required_age", regexp_replace(col("required_age"), search, replace))
clean_df.show()

+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+----------------------------+------------+------------+-------------------------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|  ccu|                   developer|discount|               genre|        header_image|initialprice|           languages|                                name|negative|              owners|positive|price|                   publisher|release_date|required_age|                    short_description|type|             website|            category|           platform|                 tag|
+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+---------------

In [0]:
print(clean_df.select('required_age').distinct().collect())

[Row(required_age='16'), Row(required_age='14'), Row(required_age='12'), Row(required_age='3'), Row(required_age='8'), Row(required_age='20'), Row(required_age='0'), Row(required_age='10'), Row(required_age='6'), Row(required_age='9'), Row(required_age='17'), Row(required_age='13'), Row(required_age='5'), Row(required_age='21'), Row(required_age='7'), Row(required_age='18'), Row(required_age='15')]


In [0]:
rules = [
    (r'\+|MA |180|35')
    ]
rule = "|".join(rules)
result = clean_df \
    .filter(col("required_age").cast('string').rlike(rule)) \
    .select('required_age') \
    .distinct() \
    .orderBy('required_age')

display(result)

required_age


In [0]:
result = clean_df.select('required_age').distinct().orderBy('required_age')
result.display()

required_age
0
10
12
13
14
15
16
17
18
20


In [0]:
rules = [
    r'#lang_',
    r'all with full audio support',
    r'full audio',
    r'not supported',
    r'text only',
    r'traditional',
    r'simplified',
    r'\- brazil',
    r'\- portugal',
    r'\(gurmukhi\)',
    r'\- latin america',
    r'\- spain',
    r'english dutch  english',
    r'slovak(?!ian)',
    r';',
    r'\[b\]\*\[/b\]',
    r'\r\n',
    r' ,',
    r',+',
    r',$',
    r"^\s*,+|,+\s*$",
    r",\s*,+"
    ]
rule = "|".join(rules)

result = clean_df \
    .filter(clean_df["languages"].rlike(rule)) \
    .select('languages')

display(result)

languages
"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean"
"English, Korean, Simplified Chinese"
"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil"
"Simplified Chinese, English, Traditional Chinese, Japanese, Korean"
"Japanese, Simplified Chinese, Traditional Chinese"
"English, Simplified Chinese, Traditional Chinese"
"English, Simplified Chinese, Traditional Chinese"
"English, Simplified Chinese"
"English, Russian, French, German, Spanish - Spain"
"English, French, Italian, German, Spanish - Spain, Arabic, Bulgarian, Czech, Danish, Dutch, Finnish, Greek, Hungarian, Japanese, Korean, Norwegian, Polish, Portuguese - Portugal, Portuguese - Brazil, Romanian, Russian, Simplified Chinese, Spanish - Latin America, Swedish, Thai, Traditional Chinese, Turkish, Ukrainian, Vietnamese"


In [0]:
from pyspark.sql.functions import regexp_replace, col

rules = [
    (r'#lang_|\(all with full audio support\)|\(text only\)|\(full audio\)|Not supported|Traditional|Simplified|\- Brazil|\- Portugal|\(gurmukhi\)|\- Latin America|\- Spain|;|\[b\]\*\[\/b\]|,\s*,', ', '),
    (r'English Dutch  English', 'English, Dutch'),
    (r'Slovak(?!ian)', 'Slovakian'),
    (r'\r\n', ', '),
    (r' ,', ','),
    (r',+', ','),
    (r',$', ''),
    (r"^\s*,+|,+\s*$", ''),
    (r",\s*,+", ","),
]

for search, replace in rules:
    clean_df = clean_df.withColumn("languages", regexp_replace(col("languages"), search, replace))
clean_df.show(truncate=False)

+-------+-------+-----+----------------------------+--------+--------------------------------------------------------------+-----------------------------------------------------------------------------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+--------+------------------------+--------+-----+-----------------------------+------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+---------------------------------------+------

In [0]:
clean_df.select(explode(split(trim(col("languages")), ","))).show()

+-----------+
|        col|
+-----------+
|    English|
|     French|
|     German|
|    Italian|
|    Spanish|
|    Chinese|
|    Chinese|
|     Korean|
|    English|
|     Korean|
|    Chinese|
|    Chinese|
|    English|
|   Japanese|
|    Chinese|
|     French|
|     German|
|    Spanish|
|    Russian|
| Portuguese|
+-----------+
only showing top 20 rows


In [0]:
rules = [
    r'#lang_',
    r'all with full audio support',
    r'full audio',
    r'not supported',
    r'text only',
    r'traditional',
    r'simplified',
    r'\- brazil',
    r'\- portugal',
    r'\(gurmukhi\)',
    r'\- latin america',
    r'\- spain',
    r'english dutch  english',
    r'slovak(?!ian)',
    r';',
    r'\[b\]\*\[/b\]',
    r'\r\n',
    r' ,',
    r',+',
    r',$',
    r"^\s*,+|,+\s*$",
    r",\s*,+"
    ]
rule = "|".join(rules)

result = clean_df \
    .select('languages') \
    .filter(clean_df["languages"].rlike(rule))
result.show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|languages                                                                                                                                                                                                                                                                |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|English, French, German, Italian, Spanish,  Chinese,  Chinese, Korean                                                                                                                              

In [0]:
result = clean_df.select('required_age').distinct().show()

+------------+
|required_age|
+------------+
|          16|
|          14|
|          12|
|           3|
|           8|
|          20|
|           0|
|          10|
|           6|
|           9|
|          17|
|          13|
|           5|
|          21|
|           7|
|          18|
|          15|
+------------+



### a) treatments

Let's clean data. First, delete unused spaces with trim method.

In [0]:
result = clean_df.select('required_age').distinct().show()

+------------+
|required_age|
+------------+
|          16|
|          14|
|          12|
|           3|
|           8|
|          20|
|           0|
|          10|
|           6|
|           9|
|          17|
|          13|
|           5|
|          21|
|           7|
|          18|
|          15|
+------------+



In [0]:
from pyspark.sql.functions import trim

clean_df = clean_df.select([trim(col(c)).alias(c) for c in clean_df.columns])
clean_df.show()


+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+----------------------------+------------+------------+-------------------------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|  ccu|                   developer|discount|               genre|        header_image|initialprice|           languages|                                name|negative|              owners|positive|price|                   publisher|release_date|required_age|                    short_description|type|             website|            category|           platform|                 tag|
+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+---------------

In [0]:
result = clean_df.select('required_age').distinct().show()

+------------+
|required_age|
+------------+
|          16|
|          14|
|          12|
|           3|
|           8|
|          20|
|           0|
|          10|
|           6|
|           9|
|          17|
|          13|
|           5|
|          21|
|           7|
|          18|
|          15|
+------------+



Then, convert numerical columns ccu, price, initialprice, discount, required_age from string to long.

In [0]:
from pyspark.sql.functions import col

string_cols = ['ccu', 'price', 'initialprice', 'discount', 'required_age']

for c in string_cols:
    clean_df = clean_df.withColumn(c, col(c).cast('long'))


In [0]:
result = clean_df.select('required_age').distinct().show()

+------------+
|required_age|
+------------+
|          16|
|           6|
|           5|
|          15|
|          10|
|          12|
|           9|
|           8|
|           7|
|          14|
|          21|
|          18|
|          17|
|          13|
|          20|
|           3|
|           0|
+------------+



Then, convert release_date from string to datetime

to_date() ne passe pas en raison de valeurs non conformes. Comptez le nombre de segments (/) pour chaque valeur distincte de 'release_date', afin de voir toutes les structures présentes (3 segments, 2 segments, etc.).

In [0]:
from pyspark.sql.functions import col, split, size

clean_df.select('release_date', size(split(col('release_date'), '/')).alias('partition_nb')) \
    .distinct() \
    .groupBy('partition_nb') \
    .count() \
    .show()

+------------+-----+
|partition_nb|count|
+------------+-----+
|           3| 3937|
|           2|   73|
|           1|    1|
+------------+-----+



Résultat clair: Trois structures existent dans 'release_date':

3 segments (3937 lignes): format complet 'yyyy/M/d'
2 segments (73 lignes): année/mois seulement, ex '2019/01'
1 segment (1 ligne): probablement juste l'année

In [0]:
from pyspark.sql.functions import to_date, col, when, split, size, concat, lit

nb_segments = size(split(col('release_date'), '/'))

clean_df = clean_df.withColumn(
    'release_date',
    when(col('release_date') == '', None)
    .when(nb_segments == 3, to_date(col('release_date'), 'yyyy/M/d'))
    .when(nb_segments == 2, to_date(concat(col('release_date'), lit('/1')), 'yyyy/M/d'))
    .when(nb_segments == 1, to_date(concat(col('release_date'), lit('/1/1')), 'yyyy/M/d'))
    .otherwise(None)
)
clean_df.select('release_date').show(5, truncate=False)

+------------+
|release_date|
+------------+
|2000-11-01  |
|2021-05-14  |
|2020-10-16  |
|2020-10-14  |
|2019-03-30  |
+------------+
only showing top 5 rows


In [0]:
price_cols = [ 'price', 'initialprice']

for c in price_cols:
    clean_df = clean_df.withColumn(c, col(c) / 100)
clean_df.select('price', 'initialprice').show(5)


+-----+------------+
|price|initialprice|
+-----+------------+
| 9.99|        9.99|
| 9.99|        9.99|
| 5.99|       19.99|
|19.99|       19.99|
| 1.99|        1.99|
+-----+------------+
only showing top 5 rows


In [0]:
result = clean_df.select('required_age').distinct().show()

+------------+
|required_age|
+------------+
|          16|
|           6|
|           5|
|          15|
|          10|
|          12|
|           9|
|           8|
|           7|
|          14|
|          21|
|          18|
|          17|
|          13|
|          20|
|           3|
|           0|
+------------+



Then, deal with latest missing values.

In [0]:
from pyspark.sql.functions import col, when, to_date, lit

for field in clean_df.schema.fields:
    c = field.name
    if field.dataType.simpleString() == 'string':
        clean_df = clean_df.withColumn(c, when(col(c) == '', 'None').otherwise(col(c)))
    elif c == 'release_date':
        clean_df = clean_df.withColumn(c, when(col(c).isNull(), to_date(lit('1900-01-01'), 'yyyy-MM-dd')).otherwise(col(c)))

clean_df.filter(col('release_date') == to_date(lit('1900-01-01'), 'yyyy-MM-dd')).show(1)

+------+------+---+--------------------+--------+-----------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+--------+--------------------+
|    id| appid|ccu|           developer|discount|      genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|   short_description|type|             website|            category|platform|                 tag|
+------+------+---+--------------------+--------+-----------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+--------+--------------------+
|102500|102500| 

In [0]:
result = clean_df.filter(col('release_date').isNull()).count()

In [0]:
result = clean_df.select('required_age').distinct().show()

+------------+
|required_age|
+------------+
|          16|
|           6|
|           5|
|          15|
|          10|
|          12|
|           9|
|           8|
|           7|
|          14|
|          21|
|          18|
|          17|
|          13|
|          20|
|           3|
|           0|
+------------+



Then, lowercase all values.

In [0]:
from pyspark.sql.functions import lower, col

string_cols = [f.name for f in clean_df.schema.fields if f.dataType.simpleString() == 'string']

clean_df = clean_df.select([lower(col(c)).alias(c) for c in string_cols] + [col(c) for c in clean_df.columns if c not in string_cols])
clean_df.show(1)

+---+-----+---------+------+--------------------+--------------------+--------------+--------+--------------------+--------+---------+--------------------+----+-------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+------------+
| id|appid|developer| genre|        header_image|           languages|          name|negative|              owners|positive|publisher|   short_description|type|website|            category|           platform|                 tag|  ccu|discount|initialprice|price|release_date|required_age|
+---+-----+---------+------+--------------------+--------------------+--------------+--------+--------------------+--------+---------+--------------------+----+-------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+------------+
| 10|   10|    valve|action|https://cdn.akama...|english, french, ...|counter-strike|    5199|10,000,000 .. 20,...|  201215|   

In [0]:
result = clean_df.select('required_age').distinct().show()

+------------+
|required_age|
+------------+
|          16|
|           6|
|           5|
|          15|
|          10|
|          12|
|           9|
|           8|
|           7|
|          14|
|          21|
|          18|
|          17|
|          13|
|          20|
|           3|
|           0|
+------------+



Endly, drop unused columns.

In [0]:
clean_df = clean_df.drop('id', 'type', 'header_image', 'website')

In [0]:
result = clean_df.select('required_age').distinct().show()

+------------+
|required_age|
+------------+
|          16|
|           6|
|           5|
|          15|
|          10|
|          12|
|           9|
|           8|
|           7|
|          14|
|          21|
|          18|
|          17|
|          13|
|          20|
|           3|
|           0|
+------------+



#TO DO compter le nombre de doublons par lignes

### b) verify clean_df dataframe

In [0]:
clean_df.createOrReplaceTempView('table') # Refresh the temporary view table in memory
result = spark.sql("select * from table limit 1")
result.show()

+-----+---------+------+--------------------+--------------+--------+--------------------+--------+---------+--------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+------------+
|appid|developer| genre|           languages|          name|negative|              owners|positive|publisher|   short_description|            category|           platform|                 tag|  ccu|discount|initialprice|price|release_date|required_age|
+-----+---------+------+--------------------+--------------+--------+--------------------+--------+---------+--------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+------------+
|   10|    valve|action|english, french, ...|counter-strike|    5199|10,000,000 .. 20,...|  201215|    valve|play the world's ...|multi-player, val...|linux, mac, windows|{"1980s":266,"199...|13990|       0|        9.99| 9.99|  2000-11-01|  

In [0]:
clean_df.printSchema()

root
 |-- appid: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: string (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- category: string (nullable = false)
 |-- platform: string (nullable = false)
 |-- tag: string (nullable = true)
 |-- ccu: long (nullable = true)
 |-- discount: long (nullable = true)
 |-- initialprice: double (nullable = true)
 |-- price: double (nullable = true)
 |-- release_date: date (nullable = true)
 |-- required_age: long (nullable = true)



In [0]:
# Number of elements in dataframe
print(f"Entries number : {clean_df.count()}")

Entries number : 55691


In [0]:
result = spark.sql("select * from table limit(5)")
result.show()

+-------+--------------------+--------------------+--------------------+--------------------+--------+--------------------+--------+--------------------+-------------------------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+------------+
|  appid|           developer|               genre|           languages|                name|negative|              owners|positive|           publisher|                    short_description|            category|           platform|                 tag|  ccu|discount|initialprice|price|release_date|required_age|
+-------+--------------------+--------------------+--------------------+--------------------+--------+--------------------+--------+--------------------+-------------------------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+------------+
|     10|               valve|              action|english

In [0]:
clean_df.select('name', 'release_date', 'developer', 'publisher', 'owners', 'ccu').sample(fraction=0.0001).show(truncate=False)

+------------------------------------+------------+---------------------+-------------------+------------------+---+
|name                                |release_date|developer            |publisher          |owners            |ccu|
+------------------------------------+------------+---------------------+-------------------+------------------+---+
|screaming loaf                      |2022-03-17  |ashumarcade          |ashumarcade        |0 .. 20,000       |0  |
|the raven - legacy of a master thief|1900-01-01  |king art             |thq nordic         |200,000 .. 500,000|1  |
|high on racing                      |2015-05-25  |silentfuture         |silentfuture       |200,000 .. 500,000|1  |
|space station loma: operations      |2017-03-17  |spielmannspiel, bison|spielmannspiel     |0 .. 20,000       |0  |
|gyrocube vr                         |2018-09-19  |swump, anders schou  |swump, anders schou|0 .. 20,000       |0  |
+------------------------------------+------------+-------------

In [0]:
result = clean_df.select('required_age').distinct().show()

+------------+
|required_age|
+------------+
|16          |
|6           |
|5           |
|15          |
|10          |
|12          |
|9           |
|8           |
|7           |
|14          |
|21          |
|18          |
|17          |
|13          |
|20          |
|3           |
|0           |
+------------+



Verify missing values.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

all_cols = clean_df.columns
string_cols = [f.name for f in clean_df.schema.fields if f.dataType.simpleString() == 'string']

clean_df.select([spark_sum((col(c).isNull() | ((col(c) == '') if c in string_cols else False)).cast('int')).alias(c) for c in all_cols])

clean_df.show()

In [0]:
result = spark.sql("select * from table where developer = 'none' or genre = 'none' or languages = 'none' or publisher = 'none' or short_description = 'none' or category = 'none'")
result.limit(1).show()

In [0]:
result = clean_df \
    .filter(col('release_date') == to_date(lit('1900-01-01'), 'yyyy-MM-dd'))
result.limit(1).show()

In [0]:

clean_df.describe().toPandas()

In [0]:
result = clean_df \
    .filter(col('release_date') != to_date(lit('1900-01-01'), 'yyyy-MM-dd')) \
    .select( \
        F.min("release_date").alias("min_release_date"),
        F.max("release_date").alias("max_release_date"))
result.show()

# TO DO Nico graph time series

## 3. Analysis at the "macro" level

Which publisher has released the most games on Steam?

In [0]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .distinct() \
    .groupBy('publisher') \
    .agg(
        count('name').alias('game_nb'),
        countDistinct('release_date').alias('release_nb')
        ) \
    .orderBy(desc('release_nb')) \
    .limit(1)
result.show()

What is Valve's position as publisher ?

In [0]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .groupBy('publisher') \
    .agg( \
        count('name').alias('game_nb'), \
        countDistinct('release_date').alias('release_nb')) \
    .withColumn('rank', row_number().over(Window
    .orderBy(desc('release_nb')))) \
    .filter(clean_df.publisher.isin(['big fish games','valve']))
result.show()

How many games on steam ?

In [0]:
result = clean_df \
    .select(clean_df['name']) \
    .distinct() \
    .count() 
print(f"Games number : {result}")

How many developers indicated on steam ?

In [0]:
result = clean_df \
    .select(clean_df['developer']) \
    .distinct() \
    .count()
print(f"Developer number : {result}")



Which developer most contribute to games on steam ?

In [0]:
result = clean_df \
    .select(clean_df['developer'],'name') \
    .distinct() \
    .groupBy('developer') \
    .agg(count('name').alias('game_nb')) \
    .orderBy(desc('game_nb')) \
    .limit(5)
result.show(truncate=False)

What are the best rated games?

In [0]:
result = clean_df \
    .select(clean_df['name'],'positive') \
    .groupBy('name') \
    .agg(F.sum('positive').alias('positive_nb')) \
    .orderBy(desc('positive_nb')) \
    .limit(10)
result.show()

In [0]:
from pyspark.sql.functions import col, try_divide

result = clean_df \
    .select(clean_df['name'],'positive', 'negative') \
    .groupBy('name') \
    .agg( \
        F.sum('positive').alias('positive_sum'), \
        F.sum('negative').alias('negative_sum') \
        ) \
    .filter((col('positive_sum') > 0) & (col('negative_sum') > 0)) \
    .withColumn('ratio', col('positive_sum') /  col('negative_sum')) \
    .orderBy(desc('positive_sum')) \
    .limit(10)
result.show(truncate=False)

In absolute value, Counter-Strike: Global Offensive has more positive advices (65,4M). Yet Terraria has a better ratio : 45,34 positive advices for 1 negative advice against 7,5 on Counter-Strike. Then, Terraria is proportianaly more liked.

Quels sont les jeux les plus utilisés au moment où le dataset a été construit ?

In [0]:
from pyspark.sql.functions import col, try_divide

result = clean_df \
    .select(clean_df['name'],'ccu') \
    .orderBy(desc('ccu'))
result.show(truncate=False)

Et quels sont les jeux les plus achetés ?

In [0]:
from pyspark.sql.functions import col, try_divide

result = clean_df \
    .groupBy('owners') \
    .agg( \
        F.count('owners').alias('owners_sum')) \
    .orderBy('owners_sum')
result.show(truncate=False)

In [0]:
from pyspark.sql.functions import col, regexp_replace, split, desc

result = clean_df \
    .select('name', 'owners', 'release_date') \
    .withColumn('owners_num', regexp_replace(split(col('owners'), ' ')[0], ',', '').cast('long')) \
    .orderBy(desc('owners_num')) \
    .drop('owners_num') \
    .limit(10)
result.show(truncate=False)

Les jeux les plus plébiscités ne sont pas forcément les plus utilisés ou les plus achetés. counter-strike: global offensive était le jeu le plus utilisé avec 874053 utilisateurs. Et dota 2 était le jeu le plus vendu avec 200 000 000 à 500 000 000 utilisateurs. counter-strike: global offensive n'a que 50 000 000 à 100 000 000 d'acheteurs. Avec la release_date, on s'aperçoit que data 2 et conter-strike offensive sont anciens.

In [0]:
result = clean_df.groupBy('owners').agg(F.count('*').alias('count')).orderBy(desc('count'))
result.show(truncate=False)

Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
from pyspark.sql.functions import (
    col, year, try_to_date, count, countDistinct
)

result = (clean_df \
    .select("name", "release_date") \
    .distinct() \
    .withColumn( \
        "release_year", \
        year(try_to_date(col("release_date"), "yyyy/M/d")) \
    ) \
    .filter(col("release_year").isNotNull()) \
    .groupBy("release_year") \
    .agg( \
        count("*").alias("release_nb"), \
        countDistinct("name").alias("name_nb") \
    ) \
    .filter((col('release_nb') > 0) & (col('name_nb') > 0)) \
    .withColumn('ratio', col('release_nb') /  col('name_nb')) \
    .orderBy(desc("release_year")) \
)

result.show()

Yes. There are years with more releases. 
- In absolute value, 2014 to 2022. There are more releases after the covid's year in 2021 with 8676. 
- In prortional value, 2015 to 2022. Releases become higher than games since 2015. Yet, releases are higher on 2020, the covid's year with 1.0011 releases for one game.  
We observe that Covid accentuated the trend which returns then at normal pace in 2022 with 7401 and 1.00094.

In [0]:
result = spark.sql("SELECT * FROM table") # filters elements from my_table where position
# show everything
result.show()

# TO DO Nico graph avec quartiles ? time series ?
How are the prizes distributed? Are there many games with a discount? regarder avec et sans mettre un count

In [0]:
result = clean_df \
    .select(clean_df['name'], 'price') \
    .orderBy('price')
display(result)

In [0]:
result = clean_df \
    .groupBy('price') \
    .agg( \
        F.count('price').alias('game_nb')) \
    .orderBy('price')
display(result)

Prices varies on a scale from 0 to 999 $, sometime of any cents. Mean price is 7,73 $ with an initial price higher on 7,93 $ (cf last run clean_df.describe() command) (cf last run clean_df.describe() command). 

In [0]:
result = clean_df \
    .groupBy(
        (F.floor(F.col('price') / 10) * 10).alias('range_price')) \
    .agg( \
        F.count('price').alias('game_nb')) \
    .orderBy('range_price')
display(result)

Prices don't change ? 

In [0]:
result = clean_df \
    .select(clean_df['publisher'], 'name', 'release_date', 'price', 'initialprice', 'discount') \
    .filter((clean_df['price'] != clean_df['initialprice']) & (clean_df['discount'] == 0)) \
    .orderBy('publisher', 'name', 'release_date') \
    .show(truncate=False)

Prices change only with discount.

Prices varies relying to discount. Le discount varie de 0 à 90 avec une valeur moyenne de 2,6. Le discount est rare.

In [0]:
result = clean_df \
    .select(clean_df['publisher'], 'developer', 'name', 'owners', 'positive', 'negative', 'release_date', 'price', 'initialprice', 'discount', 'genre', 'languages', 'required_age', 'category', 'platform', ) \
    .filter(clean_df.discount != 0 ) \
    .orderBy(desc('discount')) \
    .show()

Vérifier la répartition des jeux discount

What are the most represented languages?

In [0]:
from pyspark.sql.functions import col, explode, split, trim
result = clean_df \
    .select(explode(split(trim(col("languages")), ",")).alias('language')) \
    .groupBy("language") \
    .agg(count("*").alias("language_nb")) \
    .orderBy(desc("language_nb"))
result.display()

Chineese and spanish are overestimated. Local variants cause duplicated entries one some games.

Are there many games prohibited for children under 16/18?

Certaines valeurs posent problème : 180, 21+ et 35. A priori, les interdictions s'appliquent à des mineurs. 

Cette colonne est en strig alors qu'elle contient des integer

In [0]:
result = clean_df \
    .select('name', 'genre', 'required_age') \
    .filter(clean_df['required_age']) \
    .orderBy('required_age')
result.display()

In [0]:
result = clean_df \
    .select(clean_df['required_age'].between(16, 18)) \
    .groupBy("required_age") \
    .agg(count("*").alias("forbidden_games_nb"))
result.display()

In [0]:
clean_df.printSchema()